# Week 4 — VQC Ablation FIXED

**What changed:**  
Original Week 4 ran on **333 files** (112 `test_*` files silently mislabeled as 0).  
Fixed: **221 labeled slides** only (110 normal + 111 tumor).  
Split: train=154 (77 pos) / val=33 (17 pos) / test=34 (17 pos)

| Config | Qubits | Layers | Description |
|--------|--------|--------|--------------|
| A1 | 3 | 1 | Shallow — overfitting risk |
| A2 | 3 | 2 | Winner from old W4 — reproducibility check |
| A3 | 3 | 3 | Deep — barren plateau risk |
| A4 | 5 | 2 | Wider Hilbert space |

Settings: `max_patches=512`, `epochs=15`, `patience=5` (same speed as original W4)  
Anti-overfit: `dropout=0.5`, `weight_decay=1e-2`, `label_smoothing=0.1`


In [ ]:
# Cell 1 — Setup
import os, sys, time, json, gc
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:512'

sys.path.insert(0, '..')
from pathq.model_v2   import QuantaPathV2
from pathq.dataset_v2 import get_loaders_from_features

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT        = Path('..').resolve()
FEAT_DIR    = Path('data') / 'features_uni'
CKPT_DIR    = ROOT / 'checkpoints'
OUT_DIR     = ROOT / 'outputs'
CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

MAX_PATCHES  = 512
BATCH_SIZE   = 4
BATCH_SIZE_5Q= 2
SEED         = 42
EPOCHS       = 15
PATIENCE     = 5
LR           = 3e-5

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('CPU only')
print(f'MAX_PATCHES : {MAX_PATCHES}')
print(f'FEAT_DIR    : {FEAT_DIR.resolve()}')

In [ ]:
# Cell 2 — Load data (221 labeled slides)
train_loader, val_loader, test_loader = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = BATCH_SIZE,
    k            = 8,
    seed         = SEED,
    num_workers  = 0,
    max_patches  = MAX_PATCHES,
)
# Separate batch_size=2 loaders for 5-qubit VRAM safety
train_loader_5q, val_loader_5q, test_loader_5q = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = BATCH_SIZE_5Q,
    k            = 8,
    seed         = SEED,
    num_workers  = 0,
    max_patches  = MAX_PATCHES,
)
print(f'Train : {len(train_loader)} batches (b={BATCH_SIZE})')
print(f'Val   : {len(val_loader)} batches')
print(f'Test  : {len(test_loader)} batches')
print(f'Train_5q : {len(train_loader_5q)} batches (b={BATCH_SIZE_5Q})')

In [ ]:
# Cell 3 — Training functions
SEP  = '═' * 65
DASH = '─' * 65

def train_one(model, loader, opt, device):
    model.train()
    total, n = 0., 0
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        torch.cuda.empty_cache()
        logits, _ = model(batch)
        loss      = F.cross_entropy(logits, batch.y.view(-1), label_smoothing=0.1)
        loss_val  = loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        del logits, loss
        torch.cuda.empty_cache()
        total += loss_val; n += 1
    return total / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    probs, labels, tl, n = [], [], 0.0, 0
    for batch in loader:
        batch     = batch.to(device)
        logits, _ = model(batch)
        tl       += F.cross_entropy(logits, batch.y.view(-1)).item()
        probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        labels.extend(batch.y.view(-1).cpu().tolist())
        n += 1
    p, l  = np.array(probs), np.array(labels)
    preds = (p >= 0.5).astype(int)
    auc   = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1    = f1_score(l, preds, zero_division=0)
    tp = int(((preds == 1) & (l == 1)).sum())
    fn = int(((preds == 0) & (l == 1)).sum())
    tn = int(((preds == 0) & (l == 0)).sum())
    fp = int(((preds == 1) & (l == 0)).sum())
    return {
        'auc'        : round(auc, 6),
        'f1'         : round(f1, 6),
        'loss'       : round(tl / max(n, 1), 6),
        'sensitivity': round(tp / max(tp + fn, 1), 4),
        'specificity': round(tn / max(tn + fp, 1), 4),
    }


def run_ablation(model, tr, va, te, device, label, epochs=EPOCHS, patience=PATIENCE):
    ckpt_best   = str(CKPT_DIR / f'w4fx_{label}_best.pth')
    ckpt_latest = str(CKPT_DIR / f'w4fx_{label}_latest.pth')

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR, weight_decay=1e-2,
    )
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=1e-7
    )
    plateau_sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, min_lr=1e-7
    )

    best_auc, pat, start = 0.0, 0, 1

    if Path(ckpt_latest).exists():
        try:
            ck = torch.load(ckpt_latest, weights_only=False)
            model.load_state_dict(ck['model_state'])
            start    = ck['epoch'] + 1
            best_auc = ck['best_auc']
            for _ in range(start - 1): cosine_sched.step()
            print(f'  Resumed from epoch {start-1}  best_auc={best_auc:.4f}')
        except Exception as e:
            print(f'  Checkpoint incompatible — starting fresh. ({e})')
            start, best_auc = 1, 0.0

    print(f' {"Ep":>3} {"TrL":>8} {"VaL":>8} {"VaAUC":>7} {"VaF1":>7} {"LR":>9} {"s":>5}')
    print(DASH)

    for ep in range(start, epochs + 1):
        t0  = time.time()
        tl  = train_one(model, tr, optimizer, device)
        vm  = evaluate(model, va, device)

        cosine_sched.step()
        plateau_sched.step(vm['auc'])
        current_lr = optimizer.param_groups[0]['lr']
        flag = ''

        if vm['auc'] > best_auc:
            best_auc = vm['auc']; pat = 0; flag = '*'
            torch.save({'model_state': model.state_dict(),
                        'epoch': ep, 'best_auc': best_auc}, ckpt_best)
        else:
            pat += 1

        torch.save({'model_state': model.state_dict(),
                    'epoch': ep, 'best_auc': best_auc}, ckpt_latest)

        ow   = ' ⚠' if vm['loss'] > tl * 2.5 else ''
        secs = int(time.time() - t0)
        print(f' {ep:>3} {tl:>8.4f} {vm["loss"]:>8.4f} '
              f'{vm["auc"]:>7.4f} {vm["f1"]:>7.4f} '
              f'{current_lr:>9.2e} {secs:>4}s {flag}{ow}')

        if pat >= patience:
            print(f'  Early stop ep {ep}')
            break

        torch.cuda.empty_cache(); gc.collect()

    ck = torch.load(ckpt_best, weights_only=False)
    model.load_state_dict(ck['model_state'])
    tm = evaluate(model, te, device)
    print(DASH)
    print(f'  val AUC {best_auc:.4f} | test AUC {tm["auc"]:.4f} | gap {tm["auc"]-best_auc:+.4f}')
    return {**tm, 'val_auc': best_auc, 'gap': round(tm['auc'] - best_auc, 6)}

print('Functions loaded ✓')

In [ ]:
# Cell 4 — Ablation config table
ABLATIONS = [
    {
        'label'        : 'A1_vqc_1layer',
        'n_qubits'     : 3,
        'n_layers'     : 1,
        'desc'         : '1 layer, 3 qubits — too shallow?',
        'use_5q_loader': False,
    },
    {
        'label'        : 'A2_vqc_2layer_base',
        'n_qubits'     : 3,
        'n_layers'     : 2,
        'desc'         : '2 layers, 3 qubits — E2 winner (reproducibility)',
        'use_5q_loader': False,
    },
    {
        'label'        : 'A3_vqc_3layer',
        'n_qubits'     : 3,
        'n_layers'     : 3,
        'desc'         : '3 layers, 3 qubits — barren plateau risk?',
        'use_5q_loader': False,
    },
    {
        'label'        : 'A4_vqc_5qubit',
        'n_qubits'     : 5,
        'n_layers'     : 2,
        'desc'         : '2 layers, 5 qubits — richer Hilbert space',
        'use_5q_loader': True,
    },
]

print('Ablation configs:')
for cfg in ABLATIONS:
    print(f'  {cfg["label"]:<24} {cfg["n_qubits"]}q {cfg["n_layers"]}L  {cfg["desc"]}')

In [ ]:
# Cell 5 — Quick sanity test (1 batch per config — verify no crashes before long run)
from torch_geometric.data import Data, Batch

def quick_test(n_qubits, n_layers, label, batch_size=4):
    model = QuantaPathV2(
        use_vqc=True, n_qubits=n_qubits, vqc_layers=n_layers
    ).to(DEVICE)
    graphs = []
    for lab_y in [1, 0]:
        N = 20
        ei = torch.randint(0, N, (2, 40))
        g  = Data(
            x          = torch.randn(N, 1040).to(DEVICE),
            edge_index = ei.to(DEVICE),
            edge_attr  = torch.randn(40, 2).to(DEVICE),
            y          = torch.tensor([lab_y]).to(DEVICE),
        )
        graphs.append(g)
    b = Batch.from_data_list(graphs)
    logits, _ = model(b)
    loss = F.cross_entropy(logits, b.y.view(-1))
    loss.backward()
    del model, graphs, b
    torch.cuda.empty_cache()
    print(f'  ✓ {label}  {n_qubits}q {n_layers}L')

print('Quick smoke tests...')
for cfg in ABLATIONS:
    quick_test(cfg['n_qubits'], cfg['n_layers'], cfg['label'])
print('All configs OK — proceed with Cell 6')

In [ ]:
# Cell 6 — Run ablation (sequential — ~6-8 min/epoch at 512 patches)
ablation_results = {}

for cfg in ABLATIONS:
    print()
    print(SEP)
    print(f"  {cfg['label']}")
    print(f"  {cfg['n_qubits']} qubits × {cfg['n_layers']} layers")
    print(f"  {cfg['desc']}")
    print(SEP)

    tr, va, te = (
        (train_loader_5q, val_loader_5q, test_loader_5q)
        if cfg['use_5q_loader'] else
        (train_loader, val_loader, test_loader)
    )
    if cfg['use_5q_loader']:
        print(f'  Using batch_size={BATCH_SIZE_5Q} for 5-qubit VRAM safety')

    model = QuantaPathV2(
        use_vqc    = True,
        n_qubits   = cfg['n_qubits'],
        vqc_layers = cfg['n_layers'],
        in_dim     = 1040,
    ).to(DEVICE)

    result = run_ablation(
        model, tr, va, te, DEVICE,
        label   = cfg['label'],
        epochs  = EPOCHS,
        patience= PATIENCE,
    )
    ablation_results[cfg['label']] = {
        **result,
        'n_qubits': cfg['n_qubits'],
        'n_layers': cfg['n_layers'],
        'desc'    : cfg['desc'],
    }

    del model
    torch.cuda.empty_cache(); gc.collect()

print()
print('All ablation configs complete.')

In [ ]:
# Cell 7 — Summary table + save
# Reference: E2 quantum at 1024 patches (clean data)
E2_test_auc = 0.6851  # from outputs/E2_result.json
E2_val_auc  = 0.6507

print()
print('=' * 75)
print('WEEK 4 ABLATION — CLEAN DATA (221 slides, max_patches=512)')
print('=' * 75)
print(f'{"Config":<24} {"Q":>3} {"L":>3} {"Val AUC":>9} {"Test AUC":>9} {"F1":>7} {"Gap":>8}')
print('-' * 75)

for label, r in ablation_results.items():
    print(f"{label:<24} {r['n_qubits']:>3} {r['n_layers']:>3} "
          f"{r['val_auc']:>9.4f} {r['auc']:>9.4f} "
          f"{r['f1']:>7.4f} {r['gap']:>+8.4f}")

print('-' * 75)
print(f"{'E2 (A2 @ 1024p)':<24} {'3':>3} {'2':>3} "
      f"{E2_val_auc:>9.4f} {E2_test_auc:>9.4f} {'(ref)':>7}")
print('=' * 75)

best_label = max(ablation_results, key=lambda x: ablation_results[x]['auc'])
best       = ablation_results[best_label]
print(f'\nWinner: {best_label}  ({best["n_qubits"]}q, {best["n_layers"]}L)  test AUC={best["auc"]:.4f}')

# Save
out_data = {
    'data_note'     : 'FIXED: 221 labeled slides (removed 112 test_* unlabeled)',
    'max_patches'   : MAX_PATCHES,
    'epochs'        : EPOCHS,
    'patience'      : PATIENCE,
    'dropout'       : 0.5,
    'weight_decay'  : 1e-2,
    'label_smooth'  : 0.1,
    'E2_reference'  : {'test_auc': E2_test_auc, 'val_auc': E2_val_auc, 'note': '1024 patches'},
    'ablations'     : ablation_results,
    'winner'        : best_label,
}
with open(OUT_DIR / 'week4_ablation_fixed.json', 'w') as f:
    json.dump(out_data, f, indent=2)
print('Saved: outputs/week4_ablation_fixed.json')